In [65]:
pip install --upgrade --no-cache-dir boto3>=1.38.0 botocore>=1.38.0

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\afons\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [66]:
import boto3
import json
print(boto3.__version__)

1.43.55


In [67]:
client = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

def add_user_message(messages,text):

    user_message = {
        "role": "user",
        "content": [
            { "text": text}
        ]
    }
    messages.append(user_message)


def add_assistant_message(messages,text):

    assistant_message = {
        "role": "assistant",
        "content": [
            { "text": text}
        ]
    }
    messages.append(assistant_message)


def chat(messages, system=None,temperature=1.0, stop_sequences=[]):
    params = {"modelId": model_id, "messages": messages,"inferenceConfig":{"temperature": temperature,"stopSequences": stop_sequences}}

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]


def generate_dataset():
    prompt = """
    Generate 3 AWS-related tasks that require Python, JSON, or Regex solutions.
    
    Focus on tasks that can be solved by writing a single Python function, 
    a single JSON object, or tasks that do not require writing much code.
    
    Example output:
    [
        {
            "task": "Description of task",
            "format": "json" or "python" or "regex",
            "solution_criteria: "Must include a description of the solution criteria"
        },
        ...additional
    ]
    
    Please generate 3 objects.
    """
    messages = []

    add_user_message(messages,prompt)
    add_assistant_message(messages,"```json")
    text = chat(messages,stop_sequences=["```"])

    return json.loads(text)

### Create Eval Dataset

In [68]:
dataset = generate_dataset()

with open("dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

Draft Prompt -> Create eval dataset -> Feed through Claude -> Feed through a Grader -> Change Prompt and Repeat

### Grader

In [69]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or a commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages,stop_sequences=["```"])
    return output

In [70]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution Criteria: {test_case["solution_criteria"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages,eval_prompt)

    add_assistant_message(messages,"```json")
    eval_text = chat(messages,stop_sequences=["```"])
    return json.loads(eval_text)

In [71]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case,output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output,test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case":test_case,
        "score":score,
        "reasoning":reasoning
    }

In [72]:
import numpy as np
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = np.mean([result["score"] for result in results])

    print(average_score)
    
    return results

In [73]:
with open("dataset.json","r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

6.416666666666667


In [74]:
results

[{'output': "\nimport json\nimport sys\n\ndef extract_s3_buckets(template_content):\n    template = json.loads(template_content)\n    resources = template.get('Resources', {})\n    s3_bucket_ids = [\n        logical_id for logical_id, resource in resources.items()\n        if resource.get('Type') == 'AWS::S3::Bucket'\n    ]\n    return s3_bucket_ids\n\nif __name__ == '__main__':\n    template_content = sys.stdin.read()\n    result = extract_s3_buckets(template_content)\n    print(json.dumps(result))\n",
  'test_case': {'task': "Parse an AWS CloudFormation template and extract all resource logical IDs that are of type 'AWS::S3::Bucket'",
   'format': 'python',
   'solution_criteria': 'Must read a JSON CloudFormation template, iterate through the Resources section, filter for S3 bucket types, and return a list of logical IDs. Should handle malformed JSON gracefully.'},
  'score': 8.0,
  'reasoning': 'The solution demonstrates good Python fundamentals and correctly solves the core task fo

### Code Grader

In [75]:
import re
import ast

In [76]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [77]:
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)